In [122]:
import pandas as pd
normalised = pd.read_excel("normalised_BZK.xlsx")
# remove Unnamed columns
normalised = normalised.loc[:,~normalised.columns.str.startswith('Unnamed:')]

In [123]:
def convert_and_normalise_dates(df):
    # Identify columns ending with 'Date'
    date_columns = [col for col in df.columns if col.endswith('Date')]
    
    for col in date_columns:
        # Generate the Normalised column name
        normalised_col = col + 'Normalised'
        
        # Ensure the Normalised column exists in the DataFrame
        if normalised_col in df.columns:
            def convert_date(row):
                if pd.isna(row[normalised_col]) and pd.notna(row[col]):
                    date_value = pd.to_datetime(row[col], errors='coerce', dayfirst=True, infer_datetime_format=True)
                    if pd.notna(date_value):
                        return date_value.strftime('%Y-%m-%d')
                return row[normalised_col]
            
            # Convert the date format to yyyy-mm-dd and update the Normalised column if empty
            df[normalised_col] = df.apply(convert_date, axis=1)
    
    return df

normalised = convert_and_normalise_dates(normalised)

In [124]:
# Display only columns that contain 'Date' in their names
date_columns = [col for col in df.columns if 'Date' in col]
normalised[date_columns].head(50)

,ApplicantBirthDate,ApplicantBirthDateNormalised,VictimBirthDate,VictimBirthDateNormalised,VictimDeathDate,VictimDeaathDateNormalised
0,7.6.1818,1818-06-07,6.6.1868,1868-06-06,verst.,NaN
1,17.4.1819,1819-04-17,NaN,NaN,NaN,NaN
2,6.6.1819,1819-06-06,NaN,NaN,NaN,NaN
3,NaN,NaN,25.8.1829,1829-08-25,NaN,NaN
4,NaN,NaN,27.9.1842,1842-09-27,verstorben,NaN
5,NaN,NaN,25.12.42,1842-12-25,NaN,NaN
6,26.2.1846,1846-02-26,NaN,NaN,NaN,NaN
7,NaN,NaN,26.2.1846,1846-02-26,verst.,NaN
8,13.4.1848,1848-04-13,NaN,NaN,NaN,NaN
9,NaN,NaN,13.3.50,1850-03-13,NaN,NaN


In [125]:
#create MaritalStatus column based on ExtraRemarks1 and ExtraRemarks2

def create_marital_status(df, check_list):
    # Iterate over the rows to populate the 'Marital Status' column
    for index, row in df.iterrows():
        if str(row['ApplicantExtraRemarks1']).lower() in check_list:
            df.at[index, 'MaritalStatus'] = row['ApplicantExtraRemarks1']
        elif str(row['ApplicantExtraRemarks2']).lower() in check_list:
            df.at[index, 'MaritalStatus'] = row['ApplicantExtraRemarks2']
    
    return df

marital_list = ['led.','ledig','led','ledig.','verh','verh.','verheiratet','verw.','verw','Wwe.','Witwe','gesch.','verwitwet']
normalised = create_marital_status(normalised, marital_list)


In [126]:
normalised['MaritalStatus'].tail(20)

719          verh.
720          verh.
721          verh.
722          verh.
723          verh.
724          verh.
725           led.
726          verh.
727          verh.
728          verh.
729          verh.
730          verh.
731          verw.
732          verh.
733          verh.
734          verh.
735          verh.
736          verh.
737          verw.
738    verheiratet
Name: MaritalStatus, dtype: object

In [127]:
#create VictimDeathStatus column based on VictimExtraRemarks1 and VictimExtraRemarks2 and VictimExtraRemarks3

def create_death_status(df, check_list):
    # Iterate over the rows to populate the 'Marital Status' column
    df['VictimDeathStatus'] = ''
    for index, row in df.iterrows():
        if str(row['VictimExtraRemarks1']).lower() in check_list:
            df.at[index, 'VictimDeathStatus'] = row['VictimExtraRemarks1']
        elif str(row['VictimExtraRemarks2']).lower() in check_list:
            df.at[index, 'VictimDeathStatus'] = row['VictimExtraRemarks2']
        elif str(row['VictimExtraRemarks3']).lower() in check_list:
            df.at[index, 'VictimDeathStatus'] = row['VictimExtraRemarks3']
        elif str(row['VictimDeathDate']).lower() in check_list:
            df.at[index, 'VictimDeathStatus'] = row['VictimDeathDate']
    
    return df

death_list = ['verst.','verst','verstorben','verschollen','umgekommen']
normalised = create_death_status(normalised, death_list)

In [128]:
normalised['VictimDeathStatus'].head(50)

0         verst.
1               
2               
3               
4     verstorben
5               
6               
7         verst.
8               
9               
10              
11              
12              
13              
14              
15              
16              
17              
18              
19              
20              
21              
22              
23              
24              
25              
26              
27              
28              
29              
30              
31              
32              
33              
34              
35              
36              
37              
38              
39              
40              
41              
42              
43              
44              
45              
46              
47              
48              
49              
Name: VictimDeathStatus, dtype: object

In [129]:
existing_file_path = 'C:/Users/mahsa/Documents/Wiedergutmachung/BZK/BZK-InformationExtraction/BZK-InformationExtraction/normalised_BZK.xlsx'

with pd.ExcelWriter(existing_file_path, mode='a', engine='openpyxl', if_sheet_exists='replace') as writer:
    normalised.to_excel(writer, sheet_name='final', index=False)
